In [ ]:
import uproot
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

In [ ]:
POT_MC = 2.43007e+20
POT_OFFBEAM = 2.11e+20

In [ ]:
mode_vars = ['STANDARD','GAIN']

for RR in np.arange(1,25,1) :

    fig = plt.figure()

    gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

    print('RR interval: [',RR-1,RR,')')

    filename = 'varMC_DATA.root'
    file = uproot.open(filename)
    tree = file['tree']
    arrays = tree.arrays(
            ["_protons._dedx", 
            "_protons._rr",
            "_mu._dedx", 
            "_mu._rr"],
            library="ak"
            )
    
    #dedx_thisRR = []

    #for i in range(len(arrays)):
    #    slice = arrays[i]

    #    protons_dedx = slice['_protons._dedx'] #array di vettori di dedx, uno per protone nella slice
    #   protons_rr = slice['_protons._rr']

    #    for rr_vec,dedx_vec in zip(protons_rr,protons_dedx): #vettore rr e dedx del protone in esame nel ciclo
    #        for rr,dedx in zip(rr_vec,dedx_vec):
    #            if rr >= RR - 1 and rr < RR :
    #                dedx_thisRR.append(dedx)

    rr_flat   = ak.to_numpy(ak.flatten(arrays["_protons._rr"],   axis=None))
    dedx_flat = ak.to_numpy(ak.flatten(arrays["_protons._dedx"], axis=None))
    mask = (rr_flat >= RR - 1) & (rr_flat < RR)
    dedx_thisRR = dedx_flat[mask]
    
    h_data = ROOT.TH1D(f"h_data_rr{RR}","",100,0,30)
    h_data.Sumw2()
    data_np = np.array(dedx_thisRR)
    h_data.FillN(data_np.size,data_np,np.ones_like(data_np))
    h_data.Scale(1. / h_data.Integral("width"))

    bin_centers_DATA = []
    counts_DATA = []
    errors_DATA = []

    for bin in range(1,100 + 1):
        bin_centers_DATA.append(h_data.GetBinCenter(bin))
        counts_DATA.append(h_data.GetBinContent(bin))
        errors_DATA.append(h_data.GetBinError(bin))


    filename = 'varMC_OFFBEAM.root'
    file = uproot.open(filename)
    tree = file['tree']
    arrays = tree.arrays(
            ["_protons._dedx", 
            "_protons._rr",
            "_mu._dedx", 
            "_mu._rr"],
            library="ak"
            )
    
    rr_flat   = ak.to_numpy(ak.flatten(arrays["_protons._rr"],   axis=None))
    dedx_flat = ak.to_numpy(ak.flatten(arrays["_protons._dedx"], axis=None))
    mask = (rr_flat >= RR - 1) & (rr_flat < RR)
    dedx_thisRR_offbeam = dedx_flat[mask]
    weights_OFFBEAM_dedx = np.full(len(dedx_thisRR_offbeam), 1. / POT_OFFBEAM)

    plt.subplot(gs[0])
    plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA)

    for mode in mode_vars:

        for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

            if n == 0 and mode == 'STANDARD' : continue
            if n == 1 and mode == 'STANDARD' : sigma = ''

            filename = f'varMC_{mode}{sigma}.root'

            file = uproot.open(filename)

            tree = file['tree']
            print(tree.keys())

            arrays = tree.arrays(
            ["_protons._dedx", 
            "_protons._rr",
            "_mu._dedx", 
            "_mu._rr"],
            library="ak"
            )

            rr_flat   = ak.to_numpy(ak.flatten(arrays["_protons._rr"],   axis=None))
            dedx_flat = ak.to_numpy(ak.flatten(arrays["_protons._dedx"], axis=None))
            mask = (rr_flat >= RR - 1) & (rr_flat < RR)
            dedx_thisRR = dedx_flat[mask]
            weights_MC_dedx = np.full(len(dedx_thisRR), 1. / POT_MC)

            mc_off = np.concatenate([dedx_thisRR, dedx_thisRR_offbeam])
            weights_mc_off = np.concatenate([weights_MC_dedx, weights_OFFBEAM_dedx])
            n_mc_off, bins = np.histogram(
            mc_off,
            bins=100,
            range=(0, 30),
            weights=weights_mc_off,
            density=True
            )

            bin_centers = 0.5 * (bins[:-1] + bins[1:])
            bin_widths = np.diff(bins)

            ratio = np.zeros_like(counts_DATA)
            ratio_err = np.zeros_like(counts_DATA)

            for k in range(len(counts_DATA)):
                if n_mc_off[k] > 0:
                    ratio[k] = counts_DATA[k] / n_mc_off[k]

            sigma_n = 0
            if sigma == 'plus1sigma' : sigma_n = 1
            elif sigma == 'minus1sigma' : sigma_n = -1

            plt.subplot(gs[0])
            plt.hist([dedx_thisRR,dedx_thisRR_offbeam], weight=[weights_MC_dedx,weights_OFFBEAM_dedx],density=True,stacked=True, histtype='bar',bins=100,range=[0,30],lw=2,label=fr'{mode} {sigma_n}$\sigma$')

            plt.subplot(gs[1])

            dx = bin_centers[1] - bin_centers[0]

            edges = np.concatenate([
            [bin_centers[0] - dx/2],
            bin_centers[:-1] + dx/2,
            [bin_centers[-1] + dx/2]
            ])

            plt.stairs(ratio,edges,fill=False,lw=3,color='gray')

            plt.fill_between(
            edges[:-1],
            1,
            ratio,
            step='post',
            alpha=0.4,
            color='gray',
            lw=0
            )

            plt.axhline(1.0, color='gray', linestyle='--')

    plt.savefig(f'rr{RR}.pdf',format='pdf',bbox_inches='tight')


        
